# Module 11: Capstone, build

**Project:** build the **core of your own AI startup prototype**, alone or in a team of 2 to 4. This module is about **getting a thin version working from end to end** and **measuring it honestly**. Module 12 will harden it, price it, and present it.

**Done when:**
- your **one-page idea** is final and you have a **scope list** (must-have, nice-to-have, out of scope),
- you have a **plan that fits your time** and **interfaces your team agreed on**,
- an **end-to-end version works on real inputs**, and **someone outside your team can run it**,
- you ran your **evaluation harness** (Module 9) on it, **compared it with a simple baseline**, and recorded the results.

**How to use this notebook**
1. Save your own copy (*File > Save a copy in Drive*) and work in it with your team.
2. This is a **workbook**: it holds the templates and the checks, and your project holds the substance. Replace every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
3. **No project yet?** Set `USE_DEMO = True` in Step 1 to practise the whole workflow on a ready-made one (ticket urgency, from Module 9), then switch to your own.
4. Stuck? Use the hint ladder: the hints below each step, then the course forum. Only after your own attempt, open the solution page.

**Two rules for this module:** build the **thinnest end-to-end version first**, and **test with real inputs early**. A rough thing that works from start to finish teaches you more than a polished half.

## Step 0: Setup (given)

In [ ]:
import json, time, hashlib, random, math, re
from collections import Counter
from pathlib import Path
import numpy as np

random.seed(0); np.random.seed(0)
USE_DEMO = False      # True: practise the workflow with a ready-made demo project
print("Ready.")

## Step 1: Your project card and scope

Fill in your **project card** (refine the one-pager you wrote in Module 10) and decide what is **in** and what is **out**. Scope is the most important decision of the capstone: you have a few weeks, so a **small idea that works** beats a big one that does not.

- `PROJECT`: each text field needs at least two sentences (40 characters or more). `metric_target` is a **number between 0 and 1** that means "good enough" on your measure (for example 0.85), and `metric_name` says what is measured (for example "recall of urgent tickets").
- `TEAM`: 1 to 4 people, each with a `name` and a `role` (for example "data and labeling", "model", "API and deployment", "pitch and testing"). Roles matter even when you work alone: you wear all the hats, so write them down.
- Scope lists: `MUST_HAVE` (3 to 6 things without which there is no product), `NICE_TO_HAVE` (at least 2 things for if there is time), `OUT_OF_SCOPE` (at least 2 things you will **not** do, written down so that you do not drift into them). No item may appear in two lists.

In [ ]:
PROJECT = {
    "name": "",
    "problem": "",
    "user": "",
    "why_deep_learning": "",
    "data_source": "",
    "metric_name": "",
    "metric_target": 0.0,       # a number between 0 and 1
    "baseline": "",             # the simple approach you must beat (rules, majority class, keyword search...)
}
TEAM = []                       # for example [{"name": "Ana", "role": "data and labeling"}]
MUST_HAVE = []                  # 3 to 6 items
NICE_TO_HAVE = []               # at least 2 items
OUT_OF_SCOPE = []               # at least 2 items

### Self-check: project card and scope

In [ ]:
_text_keys = ["name", "problem", "user", "why_deep_learning", "data_source", "metric_name", "baseline"]
for _k in _text_keys:
    assert isinstance(PROJECT.get(_k), str) and len(PROJECT[_k].strip()) >= (3 if _k == "name" else 40), f"Fill in PROJECT[{_k!r}] (at least two sentences, 40 characters)."
assert isinstance(PROJECT["metric_target"], (int, float)) and 0 < PROJECT["metric_target"] <= 1, "metric_target must be a number between 0 and 1 (for example 0.85)."
assert 1 <= len(TEAM) <= 4, "TEAM must have 1 to 4 members."
assert all(isinstance(m, dict) and m.get("name") and len(m.get("role", "")) >= 5 for m in TEAM), "Each team member needs a name and a role (at least a few words)."
assert len({m["name"] for m in TEAM}) == len(TEAM), "Team member names must be different."
assert 3 <= len(MUST_HAVE) <= 6, f"MUST_HAVE has {len(MUST_HAVE)} items. Keep 3 to 6: if everything is a must, nothing is."
assert len(NICE_TO_HAVE) >= 2 and len(OUT_OF_SCOPE) >= 2, "Write at least 2 nice-to-have items and at least 2 out-of-scope items."
_all = [x.strip().lower() for x in MUST_HAVE + NICE_TO_HAVE + OUT_OF_SCOPE]
assert all(len(x) >= 10 for x in _all), "Each scope item should be a short sentence (at least 10 characters), not a single word."
assert len(set(_all)) == len(_all), "An item appears in two scope lists (or twice). Each item belongs in exactly one list."
print(f"Project card check passed: {PROJECT['name']}, team of {len(TEAM)}, {len(MUST_HAVE)} must-haves.")

## Step 2: A plan that fits

Scope on paper must fit the calendar. Write `schedule_check(plan, team_size, days_available, buffer=0.2)`. `plan` is a list of tasks, each `{"task": ..., "days": ..., "must": True or False}`. Only **must-have** tasks must fit. The function returns a dictionary with:

- `total_days`: the days of **all** tasks,
- `must_days`: the days of the tasks with `must` true,
- `per_person_days`: `must_days / team_size`,
- `needed_with_buffer`: `per_person_days * (1 + buffer)`. Plans always run late, so the **buffer** (20% by default) is not optional,
- `fits`: `True` if `needed_with_buffer <= days_available`,
- `spare_days`: `days_available - needed_with_buffer`.

In [ ]:
def schedule_check(plan, team_size, days_available, buffer=0.2):
    """Does the must-have work, shared by the team and with a safety buffer, fit in the days available?"""
    # YOUR CODE HERE
    return {}

### Self-check: schedule_check

In [ ]:
_plan = [{"task": "a", "days": 4, "must": True}, {"task": "b", "days": 6, "must": True}, {"task": "c", "days": 5, "must": False}]
_r = schedule_check(_plan, 2, 8)
assert isinstance(_r, dict) and "total_days" in _r, "schedule_check() must return a dictionary with the keys total_days, must_days, per_person_days, needed_with_buffer, fits and spare_days."
assert _r["total_days"] == 15 and _r["must_days"] == 10, f"total_days should be 15 (all tasks) and must_days 10 (only must-haves) but you got {_r['total_days']} and {_r['must_days']}."
assert abs(_r["per_person_days"] - 5.0) < 1e-9 and abs(_r["needed_with_buffer"] - 6.0) < 1e-9, f"10 must-have days shared by 2 people is 5 each, and with a 20% buffer 6.0, but you got {_r['per_person_days']} and {_r['needed_with_buffer']}."
assert _r["fits"] is True and abs(_r["spare_days"] - 2.0) < 1e-9, "6.0 days needed in 8 days available fits, with 2.0 days to spare."
_r1 = schedule_check(_plan, 1, 8)
assert _r1["fits"] is False and _r1["spare_days"] < 0, "Alone, the same plan needs 10 x 1.2 = 12 days, which does not fit in 8."
assert schedule_check(_plan, 2, 6.0)["fits"] is True and schedule_check(_plan, 2, 5.9)["fits"] is False, "Exactly enough days (6.0 needed, 6.0 available) still fits; 5.9 does not."
assert schedule_check(_plan, 2, 5.5, buffer=0.0)["fits"] is True and schedule_check(_plan, 2, 5.5, buffer=0.2)["fits"] is False, "The buffer argument must be used: 5.0 days fit in 5.5 without a buffer, but 6.0 days do not."
print("schedule_check() check passed.")

Now write **your own plan**: every task with its days, and whether it is a must-have. Include the work of Module 12 too (hardening, report, pitch) so that you do not forget it. Then set `TEAM_DAYS`: the working days that **each person** can give from now until your demo day.

In [ ]:
PLAN = []        # for example [{"task": "label 300 examples", "days": 3, "must": True}, ...]
DAYS_AVAILABLE = 0   # working days available per person until demo day

### Self-check: your plan fits

In [ ]:
assert len(PLAN) >= 5 and all(isinstance(t, dict) and {"task", "days", "must"} <= set(t) and t["days"] > 0 for t in PLAN), "PLAN needs at least 5 tasks, each with task, days (a positive number) and must (True or False)."
assert sum(1 for t in PLAN if t["must"]) >= 3 and any(not t["must"] for t in PLAN), "Mark at least 3 tasks as must-have, and at least one as nice-to-have."
assert DAYS_AVAILABLE > 0, "Set DAYS_AVAILABLE to the working days each person has until demo day."
_s = schedule_check(PLAN, len(TEAM), DAYS_AVAILABLE)
print(f"Must-have work: {_s['must_days']} days; per person {_s['per_person_days']:.1f}; with buffer {_s['needed_with_buffer']:.1f}; available {DAYS_AVAILABLE}; spare {_s['spare_days']:+.1f}")
assert _s["fits"], f"Your must-have work does not fit: it needs {_s['needed_with_buffer']:.1f} days per person with the buffer, but only {DAYS_AVAILABLE} are available. Cut scope (move something to nice-to-have) or add people or days."
print("Plan check passed: the must-have work fits, with a buffer.")

## Step 3: Interfaces your team agrees on

When several people build parts of one system, the usual failure is that the parts do not fit. The cure is to **agree the interfaces first**: for each component, **what goes in** and **what comes out**, written as plain data types, with **one owner**.

Write `check_pipeline(components)`. `components` is a list in the order the data flows, each `{"name", "input", "output", "owner"}`. Return a **list of problem messages** (an empty list if everything fits):
- for every pair of neighbours, the **output** of the first must be **exactly equal** to the **input** of the next; if not, add a message that contains **both component names** and both types,
- a component missing any of the four keys (or with an empty value) produces a message that contains its name (or its position, if it has no name).

In [ ]:
def check_pipeline(components):
    """Return a list of problems with the interfaces between components (an empty list means they fit)."""
    problems = []
    # YOUR CODE HERE
    return problems

### Self-check: check_pipeline

In [ ]:
_ok = [{"name": "clean", "input": "raw text", "output": "clean text", "owner": "Ana"}, {"name": "model", "input": "clean text", "output": "label and score", "owner": "Ben"}]
assert check_pipeline(_ok) == [], f"These components fit together, so there should be no problems, but you reported {check_pipeline(_ok)}."
_bad = [{"name": "clean", "input": "raw text", "output": "clean text", "owner": "Ana"}, {"name": "model", "input": "embedding", "output": "label", "owner": "Ben"}]
_p = check_pipeline(_bad)
assert len(_p) == 1 and "clean" in _p[0] and "model" in _p[0] and "clean text" in _p[0] and "embedding" in _p[0], f"Report one message that names both components and both types, for example 'clean outputs clean text but model expects embedding'. You got {_p}."
_missing = [{"name": "clean", "input": "raw text", "output": "clean text", "owner": ""}]
assert len(check_pipeline(_missing)) >= 1 and "clean" in " ".join(check_pipeline(_missing)), "A component with an empty owner is a problem. Mention its name."
print("check_pipeline() check passed.")

Now write **your own components** (at least 3), in the order that the data flows, with an owner from your team for each. Use short, concrete data type names such as `raw text`, `list of tokens`, `label and score`, `JSON response`.

In [ ]:
COMPONENTS = []   # for example [{"name": "preprocess", "input": "raw text", "output": "clean text", "owner": "Ana"}, ...]

### Self-check: your interfaces

In [ ]:
assert len(COMPONENTS) >= 3, "Write at least 3 components."
_probs = check_pipeline(COMPONENTS)
assert not _probs, "Your components do not fit together: " + " | ".join(_probs)
_names = {m["name"] for m in TEAM}
assert all(c["owner"] in _names for c in COMPONENTS), f"Every component's owner must be a member of TEAM: {sorted(_names)}."
print("Interfaces check passed: every part has one owner, and every output matches the next input.")

## Step 4: The thinnest end-to-end version

Now **build it**. Not the best version of each part: the **thinnest version that goes all the way from a real input to a real output**. Each part can be crude. What matters is that the whole chain runs, so that you find the surprises (formats that do not match, an input that crashes everything, a step that is far too slow) **now** and not on demo day.

Write your project's `pipeline(raw_input)` function: it takes **one raw input** (whatever your user would give: text, an image path, a question) and **returns a dictionary** with at least the key `"output"` (the answer, which must not be empty). Collect **at least 10 real inputs** (not invented to be easy!) in `REAL_INPUTS`, and write `HOW_TO_RUN`: numbered steps that let **someone outside your team** run it from a fresh start.

**Demo project:** when `USE_DEMO` is `True`, the cell below builds a ticket-urgency pipeline for you, so that you can see what a finished one looks like.

In [ ]:
def make_ticket_dataset(n=500, seed=0):
    """Synthetic support tickets. label: 'urgent' or 'normal' (about 12% urgent). Two labelers (label, label_b) and a channel (email or chat)."""
    rng = random.Random(seed)
    urgent = ["our whole site is down and customers cannot check out", "we cannot log in and our deadline is in one hour",
              "payment processing is failing for every order", "we lost all our data after the update, please help now",
              "I think someone hacked my account and changed my password", "the app crashes on startup for all our staff",
              "customers are being charged twice, this is critical", "server error 500 on every page since this morning"]
    normal = ["how do I change my billing address", "thanks for the quick reply yesterday", "can you add a dark mode in a future update",
              "what payment methods do you accept", "I would like to know how to export my report", "great product, just a small suggestion about the menu",
              "where can I find my invoice from last month", "do you offer a discount for students", "please update my email address on file",
              "how long does shipping usually take", "I forgot how to reset my password, no rush", "is there a mobile version of the dashboard",
              "login page looks different today, just curious why", "a payment reminder arrived, I wanted to confirm it is correct",
              "the site is a bit slow today but it works"]
    tricky_normal = ["this is not urgent but could you explain the invoice", "URGENT!!! just kidding, I only wanted to say thank you",
                     "no hurry at all, a question about the payment page"]
    borderline = ["login is slow for some of our team", "one customer saw a payment error once", "the export feature fails sometimes",
                  "the page crashed for me twice today"]
    openers = ["", "hi, ", "hello team, ", "good morning, ", "hey, "]
    closers = ["", " thanks", " please advise", " thank you", " regards"]
    out, seen = [], set()
    while len(out) < n:
        r = rng.random()
        if r < 0.10:
            text, lab = rng.choice(urgent), "urgent"; lab_b = lab
        elif r < 0.14:
            text = rng.choice(borderline); lab = "urgent" if rng.random() < 0.5 else "normal"; lab_b = "urgent" if rng.random() < 0.5 else "normal"
        elif r < 0.20:
            text, lab = rng.choice(tricky_normal), "normal"; lab_b = lab
        else:
            text, lab = rng.choice(normal), "normal"; lab_b = lab
        channel = "chat" if rng.random() < 0.4 else "email"
        text = rng.choice(openers) + text + rng.choice(closers)
        if channel == "email":
            text = text[0].upper() + text[1:] + "."
        else:
            text = text.lower().replace(",", "")
            words = text.split()
            if len(words) > 6 and rng.random() < 0.6: text = " ".join(words[:rng.randint(4, 6)])
        if text in seen: 
            text += " " + str(rng.randint(2, 99))
        seen.add(text)
        out.append({"text": text, "label": lab, "label_b": lab_b, "channel": channel})
    return out


def build_demo_project():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    data = make_ticket_dataset(1200, seed=0)
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000, class_weight="balanced"))
    model.fit([e["text"] for e in data], [e["label"] for e in data])
    def demo_pipeline(raw):
        if not isinstance(raw, str) or not raw.strip():
            raise ValueError("input must be a non-empty string")
        text = raw.strip()
        probs = model.predict_proba([text])[0]
        score = float(probs[list(model.classes_).index("urgent")])
        return {"output": "urgent" if score >= 0.5 else "normal", "score": round(score, 4)}
    fresh = make_ticket_dataset(500, seed=7)
    rest = [{"input": e["text"], "expected": e["label"]} for e in fresh[12:]]
    urgent = [e for e in rest if e["expected"] == "urgent"]
    normal = [e for e in rest if e["expected"] == "normal"][:len(urgent)]      # a BALANCED evaluation set: half urgent, half normal
    balanced = urgent + normal
    random.Random(0).shuffle(balanced)
    return demo_pipeline, [e["text"] for e in fresh[:12]], balanced

DEMO = build_demo_project() if USE_DEMO else None
print("demo project ready" if DEMO else "using your own project")

In [ ]:
def pipeline(raw_input):
    """Your end-to-end pipeline: one raw input in, a dictionary with a non-empty "output" out."""
    # YOUR CODE HERE
    return {"output": ""}

REAL_INPUTS = []      # at least 10 REAL inputs, as your users would send them
HOW_TO_RUN = """
YOUR CODE HERE: numbered steps (1. 2. 3.) that someone outside your team can follow.
"""

Run the **smoke test** (given): it sends every real input through your pipeline, times each call, and reports crashes. A crash on a real input is **good news today**: it is a bug you can fix now.

In [ ]:
def smoke_test(pipeline_fn, inputs):
    ok, errors, lat = 0, [], []
    for x in inputs:
        t0 = time.perf_counter()
        try:
            out = pipeline_fn(x)
            if not isinstance(out, dict) or out.get("output") in (None, ""):
                raise ValueError("the pipeline must return a dictionary with a non-empty 'output'")
            ok += 1
        except Exception as e:
            errors.append((str(x)[:70], f"{type(e).__name__}: {e}"))
        lat.append((time.perf_counter() - t0) * 1000)
    return {"n": len(inputs), "ok": ok, "errors": errors, "avg_ms": float(np.mean(lat)) if lat else 0.0, "p95_ms": float(np.percentile(lat, 95)) if lat else 0.0}

smoke = smoke_test(pipeline, REAL_INPUTS)
print(f"{smoke['ok']} of {smoke['n']} real inputs worked   (average {smoke['avg_ms']:.1f} ms, p95 {smoke['p95_ms']:.1f} ms)")
for inp, err in smoke["errors"]:
    print("  CRASH on:", inp, "->", err)

### Self-check: it works from end to end

In [ ]:
assert len(REAL_INPUTS) >= 10, f"You have {len(REAL_INPUTS)} real inputs. Collect at least 10, as your real users would send them."
assert len({str(x) for x in REAL_INPUTS}) >= 10, "Your real inputs must be different from one another."
assert smoke["ok"] == smoke["n"], f"{smoke['n'] - smoke['ok']} of your real inputs crashed or gave an empty output, for example: {smoke['errors'][0] if smoke['errors'] else ''}. Fix the pipeline until every real input works."
_h = HOW_TO_RUN.lower()
assert "your code here" not in _h and re.search(r"(^|\n)\s*1[.)]", HOW_TO_RUN) and re.search(r"(^|\n)\s*3[.)]", HOW_TO_RUN), "HOW_TO_RUN needs numbered steps (1. 2. 3.) written for someone outside your team."
assert "install" in _h or "requirements" in _h, "Say how to install the libraries (for example pip install, or a requirements.txt)."
print("End-to-end check passed: every real input works, and the steps to run it are written down.")

**One more test, which no code can do for you:** give `HOW_TO_RUN` to a person outside your team. Do **not** help them. Where do they get stuck? Fix the instructions, or the project, and write what you found:

*Your notes:*

## Step 5: Evaluate it honestly, against a baseline

A number means little on its own. "85%" could be excellent or useless. So you always compare with a **simple baseline** (the approach you wrote in `PROJECT["baseline"]`). If your model barely beats the baseline, you have learned something important, and cheaply.

You need four things:
1. **`EVAL_SET`**: at least **50** examples, each `{"input": ..., "expected": ...}`, that your model has **never trained on**. Freeze it (Module 9).
2. **`metric_fn(pred, example)`**: gives one example a score from **0 to 1** (1.0 means fully right). For classification it is just `1.0 if pred == example["expected"] else 0.0`. For other projects it can give partial credit (for example the fraction of required fields that are right).
3. **`run_capstone_eval(predict_fn, eval_set, metric_fn)`**: runs the predict function on every example and returns a dictionary with `n`, `score` (the **average** of the example scores), `failures` (a list of `{"input", "expected", "pred", "score"}` for every example scoring below 1.0), and `fingerprint` (given function below, from the eval set).
4. **`baseline_predict(input)`**: your simple baseline as a function that returns a prediction.

In [ ]:
def fingerprint_eval(eval_set):
    """A short code that changes if even one evaluation example changes."""
    blob = json.dumps([[str(e["input"]), str(e["expected"])] for e in eval_set], sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]

def metric_fn(pred, example):
    """Score one prediction from 0.0 (wrong) to 1.0 (fully right)."""
    # YOUR CODE HERE
    return 0.0

def run_capstone_eval(predict_fn, eval_set, metric_fn):
    """Run predict_fn on every example. Return {"n", "score", "failures", "fingerprint"}."""
    # YOUR CODE HERE
    return {}

### Self-check: the evaluation harness

In [ ]:
_set = [{"input": "i1", "expected": "a"}, {"input": "i2", "expected": "b"}, {"input": "i3", "expected": "a"}, {"input": "i4", "expected": "b"}]
_calls = []
def _always_a(x):
    _calls.append(x); return "a"
_m = lambda pred, ex: 1.0 if pred == ex["expected"] else 0.0
_r = run_capstone_eval(_always_a, _set, _m)
assert isinstance(_r, dict) and "n" in _r, "run_capstone_eval() must return a dictionary with the keys n, score, failures and fingerprint."
assert _r["n"] == 4 and abs(_r["score"] - 0.5) < 1e-9, f"Always answering 'a' is right on 2 of 4 examples, so score = 0.5, but you got n={_r['n']}, score={_r['score']}."
assert len(_calls) == 4, "predict_fn must be called exactly once per example."
assert [f["input"] for f in _r["failures"]] == ["i2", "i4"] and _r["failures"][0]["pred"] == "a" and _r["failures"][0]["expected"] == "b" and _r["failures"][0]["score"] == 0.0, "failures must list each example scoring below 1.0 with its input, expected, pred and score."
assert _r["fingerprint"] == fingerprint_eval(_set), "Include the fingerprint of the evaluation set."
_partial = run_capstone_eval(_always_a, _set, lambda p, ex: 0.5)
assert abs(_partial["score"] - 0.5) < 1e-9 and len(_partial["failures"]) == 4, "A partial score such as 0.5 counts as a failure (below 1.0), and the average score must use the partial scores."
_ex = _set[0]
assert metric_fn(_ex["expected"], _ex) == 1.0 or True
print("run_capstone_eval() check passed.")

Now plug in **your** evaluation set, your metric and your baseline, and run **both** your pipeline and your baseline on the **same** set. The cell is yours to fill in. (The demo fills it in for you.)

In [ ]:
EVAL_SET = []          # YOUR CODE HERE: at least 50 {"input": ..., "expected": ...}, never trained on
def baseline_predict(x):
    # YOUR CODE HERE: the simple baseline from PROJECT["baseline"]
    return ""

In [ ]:
model_report = run_capstone_eval(lambda x: pipeline(x)["output"], EVAL_SET, metric_fn)
baseline_report = run_capstone_eval(baseline_predict, EVAL_SET, metric_fn)
print(f"{PROJECT['metric_name']}")
print(f"  your model : {model_report['score']:.3f}   ({len(model_report['failures'])} failures of {model_report['n']})")
print(f"  baseline   : {baseline_report['score']:.3f}   ({len(baseline_report['failures'])} failures of {baseline_report['n']})")
print(f"  target     : {PROJECT['metric_target']}")
print(f"  eval set fingerprint: {model_report['fingerprint']}")
results = {"project": PROJECT["name"], "metric": PROJECT["metric_name"], "target": PROJECT["metric_target"],
           "model": {k: model_report[k] for k in ("n", "score", "fingerprint")}, "baseline": {k: baseline_report[k] for k in ("n", "score", "fingerprint")},
           "smoke_test": {k: smoke[k] for k in ("n", "ok", "avg_ms", "p95_ms")}}
Path("capstone_eval_build.json").write_text(json.dumps(results, indent=2))
print("Saved capstone_eval_build.json")

### Self-check: results recorded and honest

In [ ]:
assert len(EVAL_SET) >= 50, f"Your evaluation set has {len(EVAL_SET)} examples. You need at least 50."
assert all(isinstance(e, dict) and "input" in e and "expected" in e for e in EVAL_SET), "Each evaluation example needs an input and an expected output."
assert len({str(e["input"]) for e in EVAL_SET}) == len(EVAL_SET), "Duplicate inputs in the evaluation set: remove them."
assert not ({str(e["input"]) for e in EVAL_SET} & {str(x) for x in REAL_INPUTS}) or True
_ex = EVAL_SET[0]
assert metric_fn(_ex["expected"], _ex) == 1.0, "metric_fn must give a score of 1.0 to the expected answer."
assert metric_fn("@@ a clearly wrong answer @@", _ex) < 1.0, "metric_fn must give a clearly wrong answer less than 1.0."
assert model_report["fingerprint"] == baseline_report["fingerprint"] and model_report["n"] == baseline_report["n"] == len(EVAL_SET), "The model and the baseline must be evaluated on the identical evaluation set."
assert 0.0 <= model_report["score"] <= 1.0 and 0.0 <= baseline_report["score"] <= 1.0, "Scores must be between 0 and 1."
assert Path("capstone_eval_build.json").exists(), "Results were not saved."
_verdict = "BEATS" if model_report["score"] > baseline_report["score"] else "does NOT beat"
_target = "reaches" if model_report["score"] >= PROJECT["metric_target"] else "is below"
print(f"Evaluation recorded. Your model {_verdict} the baseline ({model_report['score']:.2f} vs {baseline_report['score']:.2f}) and {_target} your target ({PROJECT['metric_target']}).")

**Interpret it honestly.** Does your model beat the baseline? By how much? Is the gap big enough to justify the extra complexity? Does it reach your target? If not, is the target wrong, or the model? Write what you conclude (an honest "not yet" is a fine answer at this stage):

*Your interpretation:*

## Step 6: Look at the failures and plan next

The score says *how much* fails. The **failures** say *why*. Read them (this cell prints the first ones), look for patterns (Module 9), and write your **next steps** in order of expected value: the biggest category of failure first.

In [ ]:
print(f"{len(model_report['failures'])} failures. The first ones:\n")
for f in model_report["failures"][:15]:
    print(f"  input: {str(f['input'])[:90]}\n     expected: {f['expected']}   got: {f['pred']}\n")

In [ ]:
NEXT_STEPS = []     # YOUR CODE HERE: at least 3 prioritized items, each tied to a failure pattern you saw

### Self-check: next steps

In [ ]:
assert len(NEXT_STEPS) >= 3 and all(isinstance(s, str) and len(s.strip()) >= 30 for s in NEXT_STEPS), "Write at least 3 next steps, each a full sentence (30 characters or more) that says what you will do and why."
print("Next steps check passed. These are your plan for Module 12.")

## Stretch challenge: five users, not five opinions

Let **5 people who are not on your team** try your prototype on a real task, while you watch and **do not help**. Log for each person: did they **complete the task**, how **satisfied** were they (1 to 5), and what **issues** did they hit. Five users find most of the problems: after the first one or two, the same issues keep repeating.

Write `summarize_user_tests(tests)`. `tests` is a list of `{"completed": True or False, "satisfaction": 1 to 5, "issues": [list of short issue strings]}`. Return a dictionary with:
- `n`: the number of tests,
- `completion_rate`: the fraction that completed the task,
- `avg_satisfaction`: the average satisfaction,
- `top_issues`: a list of `(issue, count)` pairs, most frequent first (ties sorted by the issue text).

In [ ]:
def summarize_user_tests(tests):
    # YOUR CODE HERE
    return {}

### Self-check: user test summary

In [ ]:
_t = [{"completed": True, "satisfaction": 4, "issues": ["unclear button", "slow"]}, {"completed": False, "satisfaction": 2, "issues": ["slow", "no error message"]},
      {"completed": True, "satisfaction": 3, "issues": ["slow"]}, {"completed": True, "satisfaction": 5, "issues": []}]
_s = summarize_user_tests(_t)
assert isinstance(_s, dict) and "n" in _s, "summarize_user_tests() must return a dictionary with the keys n, completion_rate, avg_satisfaction and top_issues."
assert _s["n"] == 4 and abs(_s["completion_rate"] - 0.75) < 1e-9 and abs(_s["avg_satisfaction"] - 3.5) < 1e-9, f"Expected n=4, completion 0.75 and satisfaction 3.5 but got {_s}."
assert _s["top_issues"] == [("slow", 3), ("no error message", 1), ("unclear button", 1)], f"top_issues should be [('slow', 3), ('no error message', 1), ('unclear button', 1)] (most frequent first, ties by text) but you got {_s['top_issues']}."
print("summarize_user_tests() check passed.")

Run your five tests, fill in a `USER_TESTS` list like the example in the check, and print `summarize_user_tests(USER_TESTS)`. What was the **most common issue**? Did people use it the way you expected?

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-11/README.md`:
1. Your project in one sentence, and who is on the team
2. Your result against the baseline and the target, and your biggest failure pattern
3. The one thing you would cut or change in scope after what you learned

Then download this notebook (*File > Download > .ipynb*), push it to `module-11` together with `capstone_eval_build.json` (do **not** push private data), and submit the link in Moodle.